# 05 — Copula Modelling: Gaussian vs Student-t Copula

A **copula** separates the *marginal distributions* of random variables from their *joint dependence structure*. This is powerful because:

- Marginals can follow any distribution (empirical, skewed, fat-tailed).
- The dependence structure is modelled independently via the copula function.

**Sklar's Theorem** (1959) guarantees that any joint distribution can be decomposed as:

$$
F(x_1, \ldots, x_d) = C\bigl(F_1(x_1), \ldots, F_d(x_d)\bigr)
$$

where $F_i$ are marginal CDFs and $C$ is the copula.

### The 2008 Financial Crisis Connection

The **Gaussian copula** was widely used to price collateralised debt obligations (CDOs) before 2008. Its fatal flaw: **zero tail dependence**. Under a Gaussian copula, the probability of simultaneous extreme losses across assets is essentially zero — even when pairwise correlations are high. This led to systematic underestimation of systemic risk.

The **Student-t copula** corrects this by introducing a degrees-of-freedom parameter $\nu$ that controls tail dependence: lower $\nu$ means heavier tails and stronger co-movement during crises.

---

In [ ]:
import sys
sys.path.insert(0, "../src")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

from var_risk_engine.data import fetch_and_prepare
from var_risk_engine.copula import (
    fit_gaussian_copula,
    fit_t_copula,
    compare_copulas,
    copula_var,
    tail_dependence_t,
    simulate_copula,
)

%matplotlib inline

# Color palette
PRIMARY    = "#1B3A5C"
SECONDARY  = "#E8734A"
TERTIARY   = "#4CAF50"
QUATERNARY = "#9C27B0"

sns.set_theme(style="whitegrid", font_scale=1.1)
plt.rcParams["figure.figsize"] = (12, 6)
plt.rcParams["figure.dpi"] = 120

In [ ]:
# --- Fetch multi-asset return data ---
tickers = ["SPY", "QQQ", "IWM", "EFA"]
prices, returns = fetch_and_prepare(tickers, start="2019-01-01")

print(f"Assets:       {tickers}")
print(f"Period:       {returns.index[0].date()} to {returns.index[-1].date()}")
print(f"Observations: {len(returns)}")
print(f"\nReturn statistics:")
print(returns.describe().round(5).to_string())

In [ ]:
# --- Fit Gaussian Copula ---
gauss_params = fit_gaussian_copula(returns)

gauss_corr = gauss_params["corr_matrix"]

print("Gaussian Copula — Correlation Matrix")
print("=" * 50)
corr_df = pd.DataFrame(gauss_corr, index=tickers, columns=tickers)
print(corr_df.round(4).to_string())
print(f"\nObservations used: {gauss_params['n_obs']}")
print(f"Copula type:       {gauss_params['type']}")
print(f"\nKey property: Gaussian copula has ZERO tail dependence")
print(f"for any finite correlation rho in (-1, 1).")

# Visualise the correlation matrix
fig, ax = plt.subplots(figsize=(7, 6))
mask = np.triu(np.ones_like(gauss_corr, dtype=bool), k=1)
sns.heatmap(gauss_corr, mask=mask, annot=True, fmt=".3f", cmap="RdYlBu_r",
            xticklabels=tickers, yticklabels=tickers,
            vmin=-1, vmax=1, center=0,
            square=True, linewidths=0.5, ax=ax)
ax.set_title("Gaussian Copula — Correlation Matrix")
plt.tight_layout()
plt.show()

In [ ]:
# --- Fit Student-t Copula ---
t_params = fit_t_copula(returns)

t_corr = t_params["corr_matrix"]
t_nu = t_params["nu"]

print("Student-t Copula — Correlation Matrix & Degrees of Freedom")
print("=" * 55)
t_corr_df = pd.DataFrame(t_corr, index=tickers, columns=tickers)
print(t_corr_df.round(4).to_string())
print(f"\nEstimated nu (df):    {t_nu:.0f}")
print(f"Log-likelihood:       {t_params['loglikelihood']:.2f}")
print(f"Observations used:    {t_params['n_obs']}")
print(f"Copula type:          {t_params['type']}")

if t_nu <= 5:
    print(f"\n  nu = {t_nu:.0f} is low → strong tail dependence.")
    print(f"  The t-copula captures significant co-movement during extreme events.")
elif t_nu <= 10:
    print(f"\n  nu = {t_nu:.0f} is moderate → meaningful tail dependence.")
    print(f"  The t-copula captures some excess co-movement beyond Gaussian.")
else:
    print(f"\n  nu = {t_nu:.0f} is high → weak tail dependence.")
    print(f"  The t-copula is converging toward the Gaussian copula.")

# Visualise the t-copula correlation matrix
fig, ax = plt.subplots(figsize=(7, 6))
mask = np.triu(np.ones_like(t_corr, dtype=bool), k=1)
sns.heatmap(t_corr, mask=mask, annot=True, fmt=".3f", cmap="RdYlBu_r",
            xticklabels=tickers, yticklabels=tickers,
            vmin=-1, vmax=1, center=0,
            square=True, linewidths=0.5, ax=ax)
ax.set_title(f"Student-t Copula — Correlation Matrix (nu={t_nu:.0f})")
plt.tight_layout()
plt.show()

In [ ]:
# --- Compare Gaussian vs t-Copula tail dependence ---
comparison_df = compare_copulas(returns)

print("\n  Tail Dependence Comparison: Gaussian vs t-Copula")
print("  " + "=" * 75)
print(comparison_df.to_string(index=False, float_format="%.4f"))
print("  " + "=" * 75)

avg_t_tail = comparison_df["t_tail_dep"].mean()
print(f"\n  Average Gaussian tail dependence:  0.0000  (always zero)")
print(f"  Average t-Copula tail dependence:  {avg_t_tail:.4f}")
print(f"\n  The tail dependence gap means that under the t-Copula,")
print(f"  there is a {avg_t_tail*100:.1f}% chance that one asset experiences")
print(f"  an extreme loss GIVEN that another asset also does.")
print(f"  Under the Gaussian copula, this probability is zero.")

In [ ]:
# --- Side-by-side heatmaps: tail dependence (Gaussian vs t-Copula) ---
n_assets = len(tickers)

# Build symmetric tail dependence matrices
gauss_tail = np.zeros((n_assets, n_assets))
t_tail = np.zeros((n_assets, n_assets))

for i in range(n_assets):
    for j in range(n_assets):
        if i == j:
            gauss_tail[i, j] = 1.0
            t_tail[i, j] = 1.0
        else:
            rho = float(t_corr[i, j])
            gauss_tail[i, j] = 0.0  # Gaussian: always zero
            t_tail[i, j] = tail_dependence_t(rho, t_nu)

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Left: Gaussian tail dependence
mask_diag = ~np.eye(n_assets, dtype=bool)
sns.heatmap(gauss_tail, annot=True, fmt=".3f", cmap="YlOrRd",
            xticklabels=tickers, yticklabels=tickers,
            vmin=0, vmax=max(0.5, t_tail[mask_diag].max() + 0.05),
            square=True, linewidths=0.5, ax=axes[0],
            cbar_kws={"label": "Tail Dependence"})
axes[0].set_title("Gaussian Copula\nTail Dependence (always 0)", fontsize=13)

# Right: t-Copula tail dependence
sns.heatmap(t_tail, annot=True, fmt=".3f", cmap="YlOrRd",
            xticklabels=tickers, yticklabels=tickers,
            vmin=0, vmax=max(0.5, t_tail[mask_diag].max() + 0.05),
            square=True, linewidths=0.5, ax=axes[1],
            cbar_kws={"label": "Tail Dependence"})
axes[1].set_title(f"Student-t Copula (nu={t_nu:.0f})\nTail Dependence",
                   fontsize=13)

plt.suptitle("Tail Dependence: The Copula That Fooled Wall Street vs Reality",
             fontsize=14, fontweight="bold", y=1.02)
plt.tight_layout()
plt.show()

print("The Gaussian copula (left) predicts ZERO probability of simultaneous")
print("extreme losses. The t-Copula (right) captures the empirical reality")
print("that extreme events tend to cluster — the core lesson of 2008.")

In [ ]:
# --- Copula-based VaR: Gaussian vs t-Copula ---
weights = np.array([0.30, 0.30, 0.20, 0.20])
confidence_levels = [0.95, 0.975, 0.99]

var_rows = []
for cl in confidence_levels:
    v_gauss = copula_var(returns, weights, gauss_params,
                          confidence=cl, n_sims=20_000, seed=42)
    v_t = copula_var(returns, weights, t_params,
                      confidence=cl, n_sims=20_000, seed=42)
    var_rows.append({
        "confidence": cl,
        "VaR_Gaussian_Copula": v_gauss,
        "VaR_t_Copula": v_t,
        "Difference": v_t - v_gauss,
        "t_premium_pct": (v_t - v_gauss) / v_gauss * 100,
    })

var_df = pd.DataFrame(var_rows)

print("\n  Copula-Based Portfolio VaR")
print(f"  Portfolio: {dict(zip(tickers, weights))}")
print("  " + "=" * 75)
print(var_df.to_string(index=False, float_format="%.5f"))
print("  " + "=" * 75)

# Bar chart
fig, ax = plt.subplots(figsize=(10, 5))
x = np.arange(len(confidence_levels))
width = 0.35

ax.bar(x - width/2, var_df["VaR_Gaussian_Copula"], width,
       label="Gaussian Copula VaR", color=PRIMARY, edgecolor="white")
ax.bar(x + width/2, var_df["VaR_t_Copula"], width,
       label="t-Copula VaR", color=SECONDARY, edgecolor="white")

ax.set_xlabel("Confidence Level")
ax.set_ylabel("VaR (positive = loss)")
ax.set_title("Copula-Based VaR: Gaussian vs Student-t")
ax.set_xticks(x)
ax.set_xticklabels([f"{c:.1%}" for c in confidence_levels])
ax.legend(loc="upper left", frameon=True)
plt.tight_layout()
plt.show()

print(f"\nThe t-Copula VaR is consistently higher because it accounts")
print(f"for tail dependence — assets are more likely to crash together")
print(f"than the Gaussian copula predicts.")

---

## Conclusion: The Tail Dependence Gap

This notebook demonstrated the fundamental difference between Gaussian and Student-t copulas:

**The Gaussian copula's zero tail dependence** means that, no matter how high the pairwise correlation, it assigns essentially zero probability to simultaneous extreme losses across assets. This was the core modelling failure behind the 2008 financial crisis — CDO tranches rated AAA based on Gaussian copula models suffered catastrophic losses when housing markets collapsed simultaneously.

**The Student-t copula** captures the empirical reality that extreme events cluster. The estimated degrees of freedom $\nu$ directly controls the strength of tail dependence:

$$
\lambda_U = \lambda_L = 2 \, t_{\nu+1}\!\left(-\sqrt{\frac{(\nu+1)(1-\rho)}{1+\rho}}\right)
$$

Lower $\nu$ → heavier tails → stronger tail dependence → higher portfolio VaR at extreme confidence levels.

**Practical implications:**
- For **day-to-day risk** (95% VaR), the Gaussian copula may be adequate.
- For **stress testing and capital allocation** (99%+ VaR), the t-Copula's tail dependence premium becomes material.
- Regulators (Basel III/IV) implicitly acknowledge this gap by requiring stress tests that capture tail co-movement beyond what Gaussian models predict.
- The copula choice is ultimately a **model risk** decision — always compare both and understand the implications for your specific portfolio.